# ANN Optimization
### Understanding, visualizing and comparing neural-network optimizers

**Goal is to learn**
- See how an optimizer changes parameters (weights and biases).
- Compare SGD, Momentum, Nesterov, Adagrad, RMSprop, Adam and AdamW.
- Animate optimizer paths on a simple loss surface.
- Train the same small ANN while changing optimizer, learning rate, batch size and epochs.


## 1. What does an optimizer do?

Training repeatedly follows this loop:

1. The network makes predictions (forward pass).
2. A loss function measures prediction error.
3. Backpropagation calculates gradients: the direction and size of change that would increase the loss.
4. The optimizer uses those gradients to update weights and biases.

A useful mental picture: imagine standing on a hill in fog. The loss is your height. The gradient tells you the uphill direction; an optimizer decides how to take steps downhill.

The basic update is:

**theta_{t+1}=theta_t-eta(g_t)**

- theta_t: current parameter (a weight or bias) at step \(t\)
- theta_{t+1}: updated parameter
- eta: learning rate, the step-size control
- g_t: gradient of loss with respect to the parameter
- Minus sign: move opposite to the gradient, toward lower loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
np.set_printoptions(precision=4, suppress=True)

## 2. A tiny one-dimensional example

Suppose the loss is \(L(w)=(w-3)^2\). Its minimum is at \(w=3\). The derivative is \(2(w-3)\). Gradient descent repeatedly moves the weight toward 3.

In [ ]:
def loss_1d(w):
    return (w-3)**2

def grad_1d(w):
    return 2*(w-3)

w, lr = -2.0, 0.1
history = [w]
for step in range(15):
    w = w - lr*grad_1d(w)
    history.append(w)

print("Weight values:", np.round(history, 3))
plt.figure(figsize=(7,4))
ws=np.linspace(-3,7,200)
plt.plot(ws, loss_1d(ws), label="Loss curve")
plt.scatter(history, [loss_1d(x) for x in history], label="Updates")
for i in range(len(history)-1):
    plt.annotate("", (history[i+1], loss_1d(history[i+1])), (history[i], loss_1d(history[i])),
                 arrowprops=dict(arrowstyle="->"))
plt.xlabel("Weight w"); plt.ylabel("Loss"); plt.title("Gradient descent steps toward the minimum")
plt.legend(); plt.grid(alpha=.25); plt.show()

## 3. Batch, stochastic and mini-batch gradient descent

For a dataset containing \(N\) examples, the gradient can be calculated in different ways.

- **Batch Gradient Descent:** calculate the gradient using all \(N\) training examples before one update. Smooth, but each update can be expensive.
- **Stochastic Gradient Descent (SGD):** use one example for each update. Updates are frequent and noisy.
- **Mini-batch Gradient Descent:** use a small group (for example 32 examples) per update. It balances computation and stability and is the common practical choice.

The word *stochastic* means that the update is based on a randomly selected example or mini-batch, rather than always using the full dataset.

## 4. Optimizer formula guide

### SGD
\[
\theta_{t+1}=\theta_t-\eta g_t
\]
- \(\theta_t\): current weight or bias
- \(\eta\): learning rate
- \(g_t\): gradient from the current example or mini-batch

### Momentum
\[
v_t=\beta v_{t-1}+g_t,\qquad \theta_{t+1}=\theta_t-\eta v_t
\]
- \(v_t\): running direction / velocity
- \(v_{t-1}\): previous velocity
- \(\beta\): momentum factor (often 0.9); controls how much past direction is retained
- \(g_t\): current gradient
- \(\eta\): learning rate

### Nesterov Momentum (intuitive form)
\[
v_t=\beta v_{t-1}+g(\theta_t-\eta\beta v_{t-1}),\qquad
\theta_{t+1}=\theta_t-\eta v_t
\]
- The gradient is evaluated at a look-ahead position.
- \(\beta v_{t-1}\): expected movement from momentum
- \(g(\cdot)\): gradient at that look-ahead position
- \(\eta\): learning rate

### Adagrad
\[
r_t=r_{t-1}+g_t^2,\qquad
\theta_{t+1}=\theta_t-\frac{\eta}{\sqrt{r_t}+\epsilon}g_t
\]
- \(r_t\): accumulated squared gradients
- \(g_t^2\): element-wise squared gradient
- \(\epsilon\): tiny value (e.g. \(10^{-8}\)) to avoid division by zero
- Frequent large gradients build a larger denominator, reducing future step size for that parameter.

### RMSprop
\[
r_t=\rho r_{t-1}+(1-\rho)g_t^2,\qquad
\theta_{t+1}=\theta_t-\frac{\eta}{\sqrt{r_t}+\epsilon}g_t
\]
- \(\rho\): decay factor (often 0.9)
- \(r_t\): exponentially weighted average of squared gradients
- Unlike Adagrad, old squared gradients gradually lose influence.

### Adam
\[
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
\]
\[
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2
\]
\[
\hat m_t=\frac{m_t}{1-\beta_1^t},\quad
\hat v_t=\frac{v_t}{1-\beta_2^t},\quad
\theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}
\]
- \(m_t\): moving average of gradients (direction)
- \(v_t\): moving average of squared gradients (step-size scaling)
- \(\beta_1\): decay for first moment (commonly 0.9)
- \(\beta_2\): decay for second moment (commonly 0.999)
- \(t\): update number; bias correction compensates for averages starting at zero
- \(\hat m_t,\hat v_t\): bias-corrected estimates
- \(\epsilon\): numerical safety value

### AdamW
AdamW uses Adam's adaptive update and applies weight decay separately:
\[
\theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}-\eta\lambda\theta_t
\]
- \(\lambda\): weight-decay strength
- Weight decay gently discourages unnecessarily large weights; it is not the same as adding the penalty into Adam's gradient in the original Adam formulation.

## 5. Compare optimizer paths on a two-dimensional loss surface

To make the paths visible, we use a tilted, curved bowl:
\[
L(x,y)=\frac{1}{2}(x^2+12y^2)
\]
Its minimum is at (0, 0). The steep curvature in the y direction makes the path zig-zag, which helps demonstrate why momentum and adaptive learning rates can be useful. This is an illustration, not a real neural-network loss surface.

In [ ]:
def surface_grad(p):
    x,y=p
    return np.array([x, 12*y], dtype=float)

def optimize_path(name, start=np.array([-3.0, 2.0]), lr=0.08, steps=80):
    p=start.astype(float).copy()
    m=np.zeros(2); v=np.zeros(2); cache=np.zeros(2)
    beta1,beta2,eps,mu,rho=0.9,0.999,1e-8,0.9,0.9
    path=[p.copy()]
    for t in range(1,steps+1):
        g=surface_grad(p)
        if name=="SGD":
            update=g
        elif name=="Momentum":
            m=mu*m+g; update=m
        elif name=="Nesterov":
            m=mu*m+g
            update=g+mu*m
        elif name=="Adagrad":
            cache+=g*g; update=g/(np.sqrt(cache)+eps)
        elif name=="RMSprop":
            cache=rho*cache+(1-rho)*g*g
            update=g/(np.sqrt(cache)+eps)
        elif name=="Adam":
            m=beta1*m+(1-beta1)*g
            v=beta2*v+(1-beta2)*g*g
            mh=m/(1-beta1**t); vh=v/(1-beta2**t)
            update=mh/(np.sqrt(vh)+eps)
        p=p-lr*update
        path.append(p.copy())
    return np.array(path)

names=["SGD","Momentum","Nesterov","Adagrad","RMSprop","Adam"]
paths={name:optimize_path(name, lr=0.08) for name in names}
xx,yy=np.meshgrid(np.linspace(-3.5,3.5,180),np.linspace(-2.5,2.5,180))
zz=0.5*(xx**2+12*yy**2)
fig,ax=plt.subplots(figsize=(8,6))
ax.contour(xx,yy,zz,levels=25)
for name,path in paths.items():
    ax.plot(path[:,0],path[:,1],marker=".",markersize=2,label=name)
ax.scatter([0],[0],marker="*",s=180,label="Minimum")
ax.set(xlabel="x parameter",ylabel="y parameter",title="Optimizer paths on an illustrative loss surface")
ax.legend(); ax.grid(alpha=.2); plt.show()

### Animate the optimizer's movement

Choose one optimizer in the cell below. Each frame reveals the next parameter update. The contour lines represent equal loss values; the star is the minimum.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
chosen="Momentum"  # Try: SGD, Momentum, Nesterov, Adagrad, RMSprop, Adam
path=paths[chosen]
fig,ax=plt.subplots(figsize=(6,5))
ax.contour(xx,yy,zz,levels=25)
line,=ax.plot([],[],marker="o")
point,=ax.plot([],[],marker="o",markersize=8)
ax.scatter([0],[0],marker="*",s=150)
ax.set_xlim(-3.5,3.5); ax.set_ylim(-2.5,2.5)
ax.set_title(f"{chosen}: parameter updates")
ax.set_xlabel("x"); ax.set_ylabel("y")
def init():
    line.set_data([],[]); point.set_data([],[])
    return line,point
def update_frame(i):
    line.set_data(path[:i+1,0],path[:i+1,1])
    point.set_data([path[i,0]],[path[i,1]])
    return line,point
ani=FuncAnimation(fig,update_frame,frames=len(path),init_func=init,interval=100,blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

## 6. Build a small ANN and a two-moons dataset

The two-moons dataset is a simple binary classification problem with a curved boundary. It is useful because a neural network can learn a non-linear pattern.

Network:
- 2 input features
- 16 hidden neurons with ReLU
- 1 output neuron with sigmoid
- Binary cross-entropy loss

The notebook implements forward propagation, backpropagation and optimizer updates explicitly so students can inspect the process.

In [ ]:
def make_moons(n_samples=600, noise=0.15, seed=42):
    rng=np.random.default_rng(seed)
    n=n_samples//2
    t1=rng.uniform(0,np.pi,n)
    upper=np.c_[np.cos(t1),np.sin(t1)]
    t2=rng.uniform(0,np.pi,n_samples-n)
    lower=np.c_[1-np.cos(t2),0.5-np.sin(t2)]
    X=np.vstack([upper,lower])+rng.normal(0,noise,(n_samples,2))
    y=np.r_[np.zeros(n),np.ones(n_samples-n)].reshape(-1,1)
    idx=rng.permutation(n_samples)
    return X[idx],y[idx]
X,y=make_moons()
idx=np.random.default_rng(7).permutation(len(X))
cut=int(.8*len(X)); tr,te=idx[:cut],idx[cut:]
mean,std=X[tr].mean(0),X[tr].std(0)+1e-8
X_train,X_test=(X[tr]-mean)/std,(X[te]-mean)/std
y_train,y_test=y[tr],y[te]
plt.figure(figsize=(6,4))
plt.scatter(X_train[:,0],X_train[:,1],c=y_train.ravel(),s=18)
plt.title("Training dataset: two classes"); plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.show()

In [ ]:
class TinyANN:
    def __init__(self, hidden=16, seed=10):
        rng=np.random.default_rng(seed)
        self.p={"W1":rng.normal(0,np.sqrt(1), (2,hidden)),
                "b1":np.zeros((1,hidden)),
                "W2":rng.normal(0,np.sqrt(1/hidden),(hidden,1)),
                "b2":np.zeros((1,1))}
    def sigmoid(self,z):
        z=np.clip(z,-40,40); return 1/(1+np.exp(-z))
    def forward(self,X):
        z1=X@self.p["W1"]+self.p["b1"]
        a1=np.maximum(0,z1)
        z2=a1@self.p["W2"]+self.p["b2"]
        yh=self.sigmoid(z2)
        return yh,(X,z1,a1,yh)
    def loss_grads(self,X,y):
        n=len(X); yh,cache=self.forward(X)
        loss=-np.mean(y*np.log(yh+1e-8)+(1-y)*np.log(1-yh+1e-8))
        X,z1,a1,yh=cache
        dz2=(yh-y)/n
        dW2=a1.T@dz2; db2=dz2.sum(0,keepdims=True)
        da1=dz2@self.p["W2"].T
        dz1=da1*(z1>0)
        dW1=X.T@dz1; db1=dz1.sum(0,keepdims=True)
        return loss,{"W1":dW1,"b1":db1,"W2":dW2,"b2":db2}
    def predict(self,X):
        return (self.forward(X)[0]>=.5).astype(int)

In [ ]:
class Optimizer:
    def __init__(self,name,lr,params):
        self.name=name.lower(); self.lr=lr; self.t=0
        self.m={k:np.zeros_like(v) for k,v in params.items()}
        self.v={k:np.zeros_like(v) for k,v in params.items()}
        self.cache={k:np.zeros_like(v) for k,v in params.items()}
        self.beta1=.9; self.beta2=.999; self.eps=1e-8
        self.mu=.9; self.rho=.9
    def step(self,params,grads):
        self.t+=1
        for k in params:
            g=grads[k]
            if self.name=="sgd": update=g
            elif self.name=="momentum":
                self.m[k]=self.mu*self.m[k]+g; update=self.m[k]
            elif self.name=="nesterov":
                self.m[k]=self.mu*self.m[k]+g
                update=g+self.mu*self.m[k]
            elif self.name=="adagrad":
                self.cache[k]+=g*g; update=g/(np.sqrt(self.cache[k])+self.eps)
            elif self.name=="rmsprop":
                self.cache[k]=self.rho*self.cache[k]+(1-self.rho)*g*g
                update=g/(np.sqrt(self.cache[k])+self.eps)
            elif self.name in ("adam","adamw"):
                self.m[k]=self.beta1*self.m[k]+(1-self.beta1)*g
                self.v[k]=self.beta2*self.v[k]+(1-self.beta2)*g*g
                mh=self.m[k]/(1-self.beta1**self.t)
                vh=self.v[k]/(1-self.beta2**self.t)
                update=mh/(np.sqrt(vh)+self.eps)
            else: raise ValueError("Unknown optimizer")
            params[k]-=self.lr*update
            if self.name=="adamw": params[k]-=self.lr*.001*params[k]
        return params

## 7. Train and compare optimizers

Keep the dataset, network size and epoch count the same. Change only the optimizer to make a fair comparison. Results can vary slightly if you change the random seed or dataset noise.

In [ ]:
def train_ann(optimizer_name="adam", learning_rate=.01, batch_size=32, epochs=200, seed=10):
    model=TinyANN(hidden=16,seed=seed)
    opt=Optimizer(optimizer_name,learning_rate,model.p)
    rng=np.random.default_rng(123)
    history={"loss":[],"train_acc":[],"test_acc":[]}
    for epoch in range(epochs):
        order=rng.permutation(len(X_train))
        for start in range(0,len(X_train),batch_size):
            ids=order[start:start+batch_size]
            _,grads=model.loss_grads(X_train[ids],y_train[ids])
            opt.step(model.p,grads)
        loss,_=model.loss_grads(X_train,y_train)
        history["loss"].append(loss)
        history["train_acc"].append(np.mean(model.predict(X_train)==y_train))
        history["test_acc"].append(np.mean(model.predict(X_test)==y_test))
    return model,history

optimizer_names=["sgd","momentum","nesterov","adagrad","rmsprop","adam","adamw"]
results={}
for name in optimizer_names:
    model,hist=train_ann(name,learning_rate=.01,batch_size=32,epochs=200)
    results[name]=(model,hist)
    print(f"{name:10s} | loss={hist['loss'][-1]:.4f} | train acc={hist['train_acc'][-1]:.3f} | test acc={hist['test_acc'][-1]:.3f}")

In [ ]:
plt.figure(figsize=(9,5))
for name,(model,hist) in results.items():
    plt.plot(hist["loss"],label=name)
plt.xlabel("Epoch"); plt.ylabel("Training loss")
plt.title("Optimizer comparison: training loss")
plt.legend(); plt.grid(alpha=.25); plt.show()

In [ ]:
plt.figure(figsize=(9,5))
for name,(model,hist) in results.items():
    plt.plot(hist["test_acc"],label=name)
plt.xlabel("Epoch"); plt.ylabel("Test accuracy")
plt.title("Optimizer comparison: test accuracy")
plt.legend(); plt.grid(alpha=.25); plt.show()

## 8. Visualize the learned decision boundary

A decision boundary shows which class the network predicts in different regions of the feature space.

In [ ]:
def plot_boundary(model,title):
    x0_min,x0_max=X_train[:,0].min()-.5,X_train[:,0].max()+.5
    x1_min,x1_max=X_train[:,1].min()-.5,X_train[:,1].max()+.5
    gx,gy=np.meshgrid(np.linspace(x0_min,x0_max,250),np.linspace(x1_min,x1_max,250))
    grid=np.c_[gx.ravel(),gy.ravel()]
    prob=model.forward(grid)[0].reshape(gx.shape)
    plt.figure(figsize=(6,5))
    plt.contourf(gx,gy,prob,levels=np.linspace(0,1,11),alpha=.35)
    plt.scatter(X_train[:,0],X_train[:,1],c=y_train.ravel(),edgecolor="k",s=18)
    plt.title(title); plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.show()
plot_boundary(results["adam"][0],"ANN decision boundary — Adam")

## 9. Experiment: learning rate

The learning rate controls the size of each update.
- Too small: learning may be very slow.
- Reasonable: loss often decreases steadily.
- Too large: updates can jump over a good solution; loss may oscillate or diverge.

Try 0.001, 0.01 and 0.1 while keeping the optimizer, batch size and epochs fixed.

In [ ]:
learning_rates=[.001,.01,.1]
lr_results={}
for lr in learning_rates:
    _,h=train_ann("adam",learning_rate=lr,batch_size=32,epochs=150)
    lr_results[lr]=h["loss"]
plt.figure(figsize=(8,4.5))
for lr,losses in lr_results.items(): plt.plot(losses,label=f"lr={lr}")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Effect of learning rate"); plt.legend(); plt.grid(alpha=.25); plt.show()

## 10. Experiment: batch size

- Batch size 1: one training row per update; noisy but frequent updates.
- Batch size 16 or 32: small mini-batches; often a useful balance.
- Batch size 120 or full training set: fewer, steadier updates per epoch, but each update costs more.

An epoch means one complete pass through all training examples. With 480 training examples and batch size 32, there are about 15 updates per epoch.

In [ ]:
batch_sizes=[1,16,32,120]
batch_results={}
for bs in batch_sizes:
    _,h=train_ann("adam",learning_rate=.01,batch_size=bs,epochs=150)
    batch_results[bs]=h["loss"]
plt.figure(figsize=(8,4.5))
for bs,losses in batch_results.items(): plt.plot(losses,label=f"batch={bs}")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Effect of batch size"); plt.legend(); plt.grid(alpha=.25); plt.show()

## 11. Experiment: epochs and overfitting

An epoch is one full pass over the training dataset. More epochs give the network more opportunities to learn, but they do not guarantee better performance on unseen data. Watch training and test performance together. If training keeps improving while test performance worsens, the model may be overfitting.

Try 25, 100, 300 epochs and compare the curves.

## 12. Student activity

1. Run the notebook with SGD, then Momentum, then Adam.
2. Keep the learning rate and batch size fixed. Which loss curve decreases most smoothly?
3. Set Adam learning rate to 0.1. What changes?
4. Compare batch size 1, 32 and 120.
5. Find a setting that gives good test accuracy without unstable loss.
6. Explain why the optimizer with the lowest training loss is not automatically the best model.

**Important:** there is no universal best optimizer or learning rate. Results depend on the dataset, network, initialization and other settings. Adam is a strong starting point; SGD with momentum can also generalize very well and is worth comparing.